In [0]:
# spark.conf.set("spark.databricks.io.cache.enabled", False)
# spark.conf.set("spark.databricks.delta.properties.default.dataSkipingNumIndexedCols", "3")


### 1. Skew

3 Common solutions

- Adaptative Query Execution (enabled by default in Spark 3.1.)
- Filter skewed values.


## 2. Shuffles

#### Shuffle Mitigations

- Reduce network IO by using fewer larger workers
- Speed up shuffle reads an writes by using NVMe and SSDs.
- Reduce amount of shuffled data:
  - Remove unnecessary columns
  - Filter out unnecessary records preemptively.
- Denormalize datasets, especially when shuffle is rooted in a Join.

Re-evaluate the join strategy:
- Reordering the Join.
- Dynamically Switching Join Strategies.
  - Broadast Hash Join.
  - Shuffle Hash Joins:default for databricks Photon.
  - Sort-Merge Join (default for OS Spark).

  

In [0]:
from pyspark.sql.functions import rand, col, round, expr

df_transactions = (
    spark
        .range(0, 1.5*10**8, 1, 32)
        .select(
            'id',
            round(rand() * 10000, 2).alias('amount'),
            (col('id') % 10).alias('country_id'),
            (col('id') % 100).alias('store_id')
        )
        
)

df_transactions.write.mode("overwrite").saveAsTable('transactions')

In [0]:
stores_df = (
    spark
        .range(0, 99)
        .select(
            "id",
            round(rand() * 100, 0).alias('employees'),
            (col('id') % 10).alias('country_id'),
            expr('uuid()').alias('name')
        )
)

stores_df.write.mode("overwrite").saveAsTable('stores')

countries = [
    (0, "Italy"),
    (1, "Canada"),
    (2, "Mexico"),
    (3, "China"),
    (4, "Germany"),
    (5, "UK"),
    (6, "Japan"),
    (7, "Korea"),
    (8, "Australia"),
    (9, "France"),
    (10, "Spain"),
    (11, "USA"),
]

columns = ["id", "name"]
df_countries = spark.createDataFrame(data=countries, schema=columns)
df_countries.write.mode("overwrite").saveAsTable('countries')

In [0]:
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
# spark.conf.set("spark.databricks.adaptive.autoBroadcastJoinThreshold", -1)

joined_df = spark.sql("""
    SELECT
        transactions.id,
        amount,
        countries.name as country_name,
        employees,
        stores.name as store_name
        FROM transactions
        INNER JOIN stores
            ON transactions.store_id = stores.id
        LEFT JOIN countries                      
            ON transactions.country_id = countries.id 
                      
""")


# Spill
- Set spark.sql.files.maxPartitionBytes too high (default is 128 MB)
- The explode() of even small array.
- The join or crossJoin() of 2 tables which generates a lot of new rows.
- The groupBy() where the column has low cardinality.
- The countDistinct() and size(collect_set())
- Setting spark.sql.shuffle.partitions too low or wrong use of repartition();

## Spill Mitigations

- Allocate cluster with more RAM per Core.
- Address data Skew
- Manage size of Spark partitions
- Avoid expensive operations like explode().
- Reduce the amount data preemptively whenever possible.


## Serialization

- Don't use UDFs.

In [0]:
import pyspark.sql.functions as F

df = (
    spark
        .range(0, 60, 1, 1)
        .select(
            'id',
            (F.col('id') % 1000).alias('device_id'),
            (F.rand() * 100).alias('temperature_F')
        )
)

df.write.mode("overwrite").saveAsTable("device_data")

In [0]:
import pyspark.sql.types as T
import time

@udf("double")
def F_to_Celsius(f):
    time.sleep(1)
    return (f - 32) * 5 / 9

df_celsius = spark.table('device_data').withColumn("celsius", F_to_Celsius(col("temperature_F")))
df_celsius.write.mode("overwrite").saveAsTable("celsius")


In [0]:
num_partitions = 2

df_celsius = spark.table('device_data').repartition(num_partitions).withColumn("celsius", F_to_Celsius(col("temperature_F")))
df_celsius.write.mode("overwrite").saveAsTable("celsius")


In [0]:
num_partitions = 4

df_celsius = spark.table('device_data').repartition(num_partitions).withColumn("celsius", F_to_Celsius(col("temperature_F")))
df_celsius.write.mode("overwrite").saveAsTable("celsius")


In [0]:
%sql
CREATE OR REPLACE FUNCTION farh_to_cels (farh DOUBLE)
  RETURNS DOUBLE RETURN (farh - 32) * 5.0 / 9.0;

CREATE OR REPLACE TABLE celsius_sql AS
SELECT farh_to_cels(temperature_F) AS Farh_to_cels_convert FROM device_data;

SELECT * FROM celsius_sql